# release16_wf2020_foldlocal_trend_target_v1

Purpose: convert the `release16_wf2020_trend_participation_gap_v1` lagged counterfactual into a fold-local training target contract before any new retained actor is trained.

Governance:

- **Canonical baseline**: `soxlbacktest_qlib_walkforward_baseline_v1` and the current retained production comparison stack.
- **Direct comparison baseline**: `release16_joint_drawdown_release_target_contract_v1` plus the failed `release16_joint_drawdown_aux_state_head_bc_holdout_replay_v1` line.
- **Source diagnostic**: `release16_wf2020_trend_participation_gap_v1`.
- **Role**: `diagnose` and target-readiness handoff; no actor retention or promotion is possible in this card.
- **Target window**: fold-local train rows only: `wf_2020_train`, `wf_2022_train`, and `wf_2023_2024_train`. Eval windows remain excluded.
- **Changed layer**: `training_target`, `model_selection`, and BC/checkpoint handoff.
- **External reference**: Qlib-style repeatable evaluation, Probability of Backtest Overfitting promotion discipline, time-series momentum/trend participation, and volatility-managed trend guards.
- **Hypothesis**: the wf2020 lagged counterfactual can become a non-leaking fold-local participation target if it has train-window feature support, valid target lift semantics, and healthy policy action labels.
- **Acceptance criteria**: zero eval overlap, positive trend target rows in `wf_2020_train`, target lift audit passes, actual `sparse_release_bc_ppo_screen_v1` action-label distribution passes BC readiness, and promotion stays blocked until an official three-fold replay.
- **Expected trade-off**: more high-trend participation labels can improve wf2020 upside but may increase drawdown unless the downstream replay preserves the `20%` MaxDD guard.
- **Failure mode**: target rows are absent in `wf_2020_train`, labels collapse to one action, or the OOS-derived recipe overfits and fails official walk-forward replay.
- **Comparison-table row key**: `release16_wf2020_foldlocal_trend_target_v1`.

Historical duplicate check:

- Closest success: `release16_joint_drawdown_release_target_contract_v1` passed fold-local scope, target lift, and BC readiness, but it did not repair the wf2020 trend participation gap.
- Closest failure: `trend_participation_foldlocal_target_v1` created trend labels for later folds, but it did not transfer the release16 wf2020 lagged counterfactual and used a bucket approximation for label health.
- Why this is not a duplicate: this notebook uses the release16 joint target contract, the exact best lagged wf2020 counterfactual recipe, and the actual `sparse_release_bc_ppo_screen_v1` policy action mapping for BC readiness.

In [1]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import json
import math
import os
import sys

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'AGENTS.md').exists())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.state_objective_training_targets import (
    TrendParticipationTargetConfig,
    apply_trend_participation_target_gate,
    audit_state_objective_target_lift,
    audit_state_objective_target_scope,
    score_state_objective_bc_readiness,
    summarize_state_objective_bc_readiness,
    summarize_state_objective_target_lift_audit,
    summarize_state_objective_target_scope_audit,
)
from src.soxl_quant.config import load_profile
from src.soxl_quant.data import build_feature_frame, flatten_feature_frame, init_qlib
from src.soxl_quant.policy.policy_builder import (
    build_state_objective_warm_start_dataset_from_frame,
    state_objective_target_position_from_row,
)
from src.soxl_quant.settings import QLIB_DATA_PATH

EXPERIMENT = 'release16_wf2020_foldlocal_trend_target_v1'
CARD = ROOT / 'runtime' / 'model_selection' / 'card' / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)

SOURCE_TARGET_CARD = ROOT / 'runtime/model_selection/card/release16_joint_drawdown_release_target_contract_v1'
SOURCE_TARGET_CONTRACT = SOURCE_TARGET_CARD / 'joint_target_contract.csv'
SOURCE_GAP_CARD = ROOT / 'runtime/model_selection/card/release16_wf2020_trend_participation_gap_v1'
SOURCE_GAP_SUMMARY = SOURCE_GAP_CARD / 'summary.json'
PROFILE_NAME = 'sparse_release_bc_ppo_screen_v1'
TARGET_SCOPE = 'fold_local'
TARGET_MAX_DD_PCT = 20.0
MIN_WF2020_TREND_ROWS = 25
MAX_TREND_TARGET_RATIO = 0.30
OBJECTIVE_WEIGHT_BONUS = 4.0

SIDE_CARS = [
    'trend_target_config.json',
    'counterfactual_transfer_audit.json',
    'trend_target_contract.csv',
    'selected_trend_target_contract.csv',
    'trend_target_counts.csv',
    'feature_overlap_audit.csv',
    'target_scope_audit.csv',
    'target_lift_audit.csv',
    'target_lift_summary.csv',
    'target_position_distribution.csv',
    'profile_label_comparison.csv',
    'action_label_counts.csv',
    'bc_readiness_scores.csv',
    'readiness_scores.csv',
    'selection.json',
    'summary.json',
    'conclusion.md',
]

print('card', CARD)
print('source_target_contract', SOURCE_TARGET_CONTRACT)
print('source_gap_summary', SOURCE_GAP_SUMMARY)

card /Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/runtime/model_selection/card/release16_wf2020_foldlocal_trend_target_v1
source_target_contract /Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/runtime/model_selection/card/release16_joint_drawdown_release_target_contract_v1/joint_target_contract.csv
source_gap_summary /Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/runtime/model_selection/card/release16_wf2020_trend_participation_gap_v1/summary.json


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        value = float(value)
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return value.strftime('%Y-%m-%d')
    if pd.isna(value) if not isinstance(value, (dict, list, tuple, set)) else False:
        return None
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True) + '\n', encoding='utf-8')


def bool_series(frame: pd.DataFrame, column: str, *, default: bool = False) -> pd.Series:
    if column not in frame.columns:
        return pd.Series(default, index=frame.index)
    values = frame[column]
    if values.dtype == object:
        return values.fillna(default).astype(str).str.lower().isin(['1', 'true', 'yes', 'y'])
    return values.fillna(default).astype(bool)


def derive_folds(targets: pd.DataFrame) -> list[dict[str, str]]:
    required = ['fold', 'train_start', 'train_end', 'eval_start', 'eval_end']
    missing = sorted(set(required) - set(targets.columns))
    if missing:
        raise ValueError(f'source targets missing fold metadata: {missing}')
    folds: list[dict[str, str]] = []
    for fold, group in targets.groupby('fold', sort=True):
        unique = group[required].drop_duplicates()
        if len(unique) != 1:
            raise ValueError(f'fold {fold} has inconsistent boundaries')
        row = unique.iloc[0]
        folds.append({
            'name': str(fold),
            'train_start': str(row['train_start']),
            'train_end': str(row['train_end']),
            'eval_start': str(row['eval_start']),
            'eval_end': str(row['eval_end']),
        })
    return folds


def trend_config_from_counterfactual(row: dict[str, Any], *, enabled: bool) -> TrendParticipationTargetConfig:
    return TrendParticipationTargetConfig(
        enabled=bool(enabled),
        min_target_position=float(row['min_target']),
        max_target_position=float(row['max_target']),
        max_teacher_position=float(row['max_base_position']),
        min_anchor_gap_180=float(row['min_anchor_gap_180']),
        min_mom_63=float(row['min_mom_63']),
        min_trend_slope_63=float(row['min_trend_slope_63']),
        min_trend_gap_63=float(row['min_trend_gap_63']),
        max_vix_percentile_252=float(row['max_vix_percentile_252']),
        max_realized_vol_63=float(row['max_realized_vol_63']),
        min_drawdown_63=float(row['min_drawdown_63']),
        objective_weight_bonus=float(OBJECTIVE_WEIGHT_BONUS),
        exclude_downtrend=True,
    )


def target_position_series(targets: pd.DataFrame) -> pd.Series:
    required = ['teacher_position', 'rebound_target_position', 'risk_target_position', 'is_rebound_cluster', 'is_downtrend_state']
    frame = targets.copy()
    for col in required[:3]:
        frame[col] = pd.to_numeric(frame[col], errors='coerce').fillna(0.0).clip(0.0, 1.0)
    for col in required[3:]:
        frame[col] = bool_series(frame, col)
    return pd.Series(
        [state_objective_target_position_from_row(row) for row in frame[required].to_dict('records')],
        index=frame.index,
        dtype=float,
    )


def position_distribution(targets: pd.DataFrame, *, candidate: str) -> pd.DataFrame:
    frame = targets.copy()
    frame['state_objective_target_position'] = target_position_series(frame)
    frame['position_bucket_5pct'] = np.round(frame['state_objective_target_position'] / 0.05).astype(int).clip(0, 20)
    rows: list[dict[str, Any]] = []
    for fold, group in frame.groupby('fold', sort=True):
        counts = group['position_bucket_5pct'].value_counts(normalize=True).sort_index()
        rows.append({
            'candidate': candidate,
            'fold': str(fold),
            'rows': int(len(group)),
            'unique_position_buckets': int(group['position_bucket_5pct'].nunique()),
            'dominant_position_bucket_share': float(counts.max()) if not counts.empty else 1.0,
            'mean_target_position': float(group['state_objective_target_position'].mean()),
            'max_target_position': float(group['state_objective_target_position'].max()),
            'trend_target_rows': int(bool_series(group, 'trend_participation_target_state').sum()),
        })
    return pd.DataFrame(rows)


def actual_policy_action_distribution(
    features: pd.DataFrame,
    targets: pd.DataFrame,
    profile: Any,
    *,
    candidate: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows: list[dict[str, Any]] = []
    count_rows: list[dict[str, Any]] = []
    all_actions: list[int] = []

    for fold, group in targets.groupby('fold', sort=True):
        label = str(fold)
        try:
            _obs, actions = build_state_objective_warm_start_dataset_from_frame(
                features,
                group.copy(),
                profile,
                sample_size=0,
            )
            action_series = pd.Series(actions, dtype='int64')
            counts = action_series.value_counts(normalize=True).sort_index()
            for action, share in counts.items():
                count_rows.append({
                    'candidate': candidate,
                    'profile': label,
                    'action': int(action),
                    'count': int((action_series == int(action)).sum()),
                    'share': float(share),
                })
            rows.append({
                'candidate': candidate,
                'profile': label,
                'samples': int(len(action_series)),
                'unique_actions': int(action_series.nunique()),
                'dominant_action_share': float(counts.max()) if not counts.empty else 1.0,
                'action_entropy': float(-(counts * np.log(counts + 1e-12)).sum()) if not counts.empty else 0.0,
                'label_error': '',
            })
            all_actions.extend(action_series.tolist())
        except Exception as exc:  # readiness report should show the failure instead of hiding it
            rows.append({
                'candidate': candidate,
                'profile': label,
                'samples': 0,
                'unique_actions': 0,
                'dominant_action_share': 1.0,
                'action_entropy': 0.0,
                'label_error': str(exc),
            })

    if all_actions:
        action_series = pd.Series(all_actions, dtype='int64')
        counts = action_series.value_counts(normalize=True).sort_index()
        for action, share in counts.items():
            count_rows.append({
                'candidate': candidate,
                'profile': 'all_folds_actual_policy_mapping',
                'action': int(action),
                'count': int((action_series == int(action)).sum()),
                'share': float(share),
            })
        rows.append({
            'candidate': candidate,
            'profile': 'all_folds_actual_policy_mapping',
            'samples': int(len(action_series)),
            'unique_actions': int(action_series.nunique()),
            'dominant_action_share': float(counts.max()) if not counts.empty else 1.0,
            'action_entropy': float(-(counts * np.log(counts + 1e-12)).sum()) if not counts.empty else 0.0,
            'label_error': '',
        })

    return pd.DataFrame(rows), pd.DataFrame(count_rows)


def trend_count_table(targets: pd.DataFrame, *, candidate: str, required_features: list[str]) -> pd.DataFrame:
    frame = targets.copy()
    rows: list[dict[str, Any]] = []
    for fold, group in frame.groupby('fold', sort=True):
        trend = bool_series(group, 'trend_participation_target_state')
        signal = bool_series(group, 'trend_participation_signal_ready')
        guard = bool_series(group, 'trend_participation_guard_ready')
        rows.append({
            'candidate': candidate,
            'fold': str(fold),
            'rows': int(len(group)),
            'feature_complete_rows': int(group[required_features].notna().all(axis=1).sum()) if required_features else int(len(group)),
            'signal_ready_rows': int(signal.sum()),
            'guard_ready_rows': int(guard.sum()),
            'trend_target_rows': int(trend.sum()),
            'trend_target_ratio': float(trend.mean()) if len(group) else 0.0,
            'mean_teacher_position': float(pd.to_numeric(group['teacher_position'], errors='coerce').mean()),
            'mean_rebound_target_position': float(pd.to_numeric(group['rebound_target_position'], errors='coerce').mean()),
            'mean_risk_target_position': float(pd.to_numeric(group['risk_target_position'], errors='coerce').mean()),
        })
    return pd.DataFrame(rows)

In [3]:
source_targets = pd.read_csv(SOURCE_TARGET_CONTRACT)
source_targets['datetime'] = pd.to_datetime(source_targets['datetime']).dt.normalize()
folds = derive_folds(source_targets)
gap_summary = json.loads(SOURCE_GAP_SUMMARY.read_text(encoding='utf-8'))
best_counterfactual = gap_summary['best_lagged_counterfactual']

profile = load_profile(PROFILE_NAME)
init_qlib(QLIB_DATA_PATH)
feature_start = min(fold['train_start'] for fold in folds)
feature_end = max(fold['train_end'] for fold in folds)
feature_frame, _price = build_feature_frame(
    feature_start,
    feature_end,
    feature_packs=tuple(profile.feature.packs),
    profile_name=profile.name,
)
features = flatten_feature_frame(feature_frame).reset_index()
features['datetime'] = pd.to_datetime(features['datetime']).dt.normalize()

input_profile = pd.DataFrame([
    {
        'source_target_rows': int(len(source_targets)),
        'source_target_min_date': source_targets['datetime'].min().strftime('%Y-%m-%d'),
        'source_target_max_date': source_targets['datetime'].max().strftime('%Y-%m-%d'),
        'feature_rows': int(len(features)),
        'feature_min_date': features['datetime'].min().strftime('%Y-%m-%d'),
        'feature_max_date': features['datetime'].max().strftime('%Y-%m-%d'),
        'profile': PROFILE_NAME,
        'feature_packs': ','.join(profile.feature.packs),
        'source_gap_decision': gap_summary.get('decision'),
        'source_gap_blocker': gap_summary.get('primary_blocker'),
        'source_gap_next_mechanism': gap_summary.get('next_mechanism'),
    }
])
input_profile.to_csv(CARD / 'input_profile.csv', index=False)
input_profile

[72228:MainThread](2026-06-15 12:46:31,997) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[72228:MainThread](2026-06-15 12:46:32,421) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[72228:MainThread](2026-06-15 12:46:32,421) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/data/qlib/soxl')}


,source_target_rows,source_target_min_date,source_target_max_date,feature_rows,feature_min_date,feature_max_date,profile,feature_packs,source_gap_decision,source_gap_blocker,source_gap_next_mechanism
0,3731,2010-03-11,2022-12-30,2973,2011-03-11,2022-12-30,sparse_release_bc_ppo_screen_v1,,release16_wf2020_trend_participation_gap_no_pr...,official_replay_required_for_lagged_counterfac...,convert_visible_wf2020_trend_participation_gap...


In [4]:
base_config = trend_config_from_counterfactual(best_counterfactual, enabled=False)
trend_config = trend_config_from_counterfactual(best_counterfactual, enabled=True)
required_trend_features = [
    'anchor_gap_180',
    'mom_63',
    'trend_slope_63',
    'trend_gap_63',
    'vix_percentile_252',
    'realized_vol_63',
    'drawdown_63',
]

base_contract = apply_trend_participation_target_gate(source_targets, features, base_config)
base_contract['candidate'] = 'base_release16_joint_contract'
base_contract['target_recipe'] = 'base_release16_joint_contract'
base_contract['source_counterfactual_oos_informed'] = False
base_contract['oos_config_requires_official_replay'] = False

trend_contract = apply_trend_participation_target_gate(source_targets, features, trend_config)
trend_contract['candidate'] = 'wf2020_lagged_trend_participation_floor'
trend_contract['target_recipe'] = 'wf2020_lagged_trend_participation_floor'
trend_contract['source_counterfactual_oos_informed'] = True
trend_contract['source_counterfactual_card'] = 'release16_wf2020_trend_participation_gap_v1'
trend_contract['source_counterfactual_mode'] = str(best_counterfactual.get('mode'))
trend_contract['source_counterfactual_relative_return_pp'] = float(best_counterfactual['relative_return_pp'])
trend_contract['source_counterfactual_max_dd_pct'] = float(best_counterfactual['max_dd_pct'])
trend_contract['oos_config_requires_official_replay'] = True
trend_contract['counterfactual_equity_guard_not_transferred_to_foldlocal_target'] = True
trend_contract['downstream_replay_max_equity_drawdown_guard'] = best_counterfactual.get('max_equity_drawdown')

candidate_targets = {
    'base_release16_joint_contract': base_contract,
    'wf2020_lagged_trend_participation_floor': trend_contract,
}

write_json(CARD / 'trend_target_config.json', {
    'experiment': EXPERIMENT,
    'profile': PROFILE_NAME,
    'source_gap_card': 'release16_wf2020_trend_participation_gap_v1',
    'source_target_contract': 'release16_joint_drawdown_release_target_contract_v1/joint_target_contract.csv',
    'best_lagged_counterfactual': best_counterfactual,
    'trend_participation_target_config': trend_config.__dict__,
    'target_scope': TARGET_SCOPE,
    'target_max_dd_pct': TARGET_MAX_DD_PCT,
    'min_wf2020_trend_rows': MIN_WF2020_TREND_ROWS,
    'max_trend_target_ratio': MAX_TREND_TARGET_RATIO,
})

write_json(CARD / 'counterfactual_transfer_audit.json', {
    'source_counterfactual_is_oos_informed': True,
    'fold_local_label_generation': True,
    'eval_rows_used_as_labels': False,
    'promotion_allowed_from_this_card': False,
    'actor_retention_allowed_from_this_card': False,
    'official_three_fold_replay_required': True,
    'counterfactual_equity_guard_transferred_to_target': False,
    'reason_equity_guard_not_transferred': 'max_equity_drawdown is path-dependent in the OOS counterfactual and is kept as a downstream replay guard, not a fold-local row label.',
    'downstream_replay_max_equity_drawdown_guard': best_counterfactual.get('max_equity_drawdown'),
})

trend_contract.to_csv(CARD / 'trend_target_contract.csv', index=False)
print('trend_target_rows', int(bool_series(trend_contract, 'trend_participation_target_state').sum()))
trend_count_table(trend_contract, candidate='wf2020_lagged_trend_participation_floor', required_features=required_trend_features)

trend_target_rows 631


,candidate,fold,rows,feature_complete_rows,signal_ready_rows,guard_ready_rows,trend_target_rows,trend_target_ratio,mean_teacher_position,mean_rebound_target_position,mean_risk_target_position
0,wf2020_lagged_trend_participation_floor,wf_2020_train,2470,2217,992,1228,357,0.144534,0.413078,0.574024,0.390904
1,wf2020_lagged_trend_participation_floor,wf_2022_train,505,505,299,264,136,0.269307,0.208183,0.502538,0.330257
2,wf2020_lagged_trend_participation_floor,wf_2023_2024_train,756,756,310,271,138,0.182540,0.160822,0.429381,0.301777


In [5]:
scope_rows: list[pd.DataFrame] = []
lift_rows: list[pd.DataFrame] = []
lift_summary_rows: list[dict[str, Any]] = []
label_rows: list[pd.DataFrame] = []
action_count_rows: list[pd.DataFrame] = []
bc_rows: list[pd.DataFrame] = []
position_rows: list[pd.DataFrame] = []
count_rows: list[pd.DataFrame] = []
readiness_rows: list[dict[str, Any]] = []

for candidate, targets in candidate_targets.items():
    scope_audit = audit_state_objective_target_scope(targets, folds=folds)
    scope_audit.insert(0, 'candidate', candidate)
    scope_summary = summarize_state_objective_target_scope_audit(scope_audit.drop(columns=['candidate']))

    lift_audit = audit_state_objective_target_lift(
        targets,
        group_cols=('fold',),
        baseline_risk_position_col='risk_target_position_original',
    )
    lift_audit.insert(0, 'candidate', candidate)
    lift_summary = summarize_state_objective_target_lift_audit(lift_audit.drop(columns=['candidate']))
    lift_summary_rows.append({'candidate': candidate, **lift_summary})

    profile_dist, action_counts = actual_policy_action_distribution(
        features,
        targets,
        profile,
        candidate=candidate,
    )
    bc_scores = score_state_objective_bc_readiness(
        profile_dist[['profile', 'samples', 'unique_actions', 'dominant_action_share', 'action_entropy']],
        target_scope=TARGET_SCOPE,
    )
    bc_scores.insert(0, 'candidate', candidate)
    all_fold_scores = bc_scores[bc_scores['profile'].eq('all_folds_actual_policy_mapping')]
    bc_summary = summarize_state_objective_bc_readiness(all_fold_scores.drop(columns=['candidate']))

    counts = trend_count_table(targets, candidate=candidate, required_features=required_trend_features)
    trend_rows = int(counts['trend_target_rows'].sum())
    total_rows = int(counts['rows'].sum())
    trend_ratio = float(trend_rows / max(total_rows, 1))
    fold_counts = dict(zip(counts['fold'].astype(str), counts['trend_target_rows'].astype(int)))
    wf2020_trend_rows = int(fold_counts.get('wf_2020_train', 0))
    trend_gate = bool(
        candidate != 'base_release16_joint_contract'
        and trend_rows > 0
        and wf2020_trend_rows >= int(MIN_WF2020_TREND_ROWS)
        and trend_ratio <= float(MAX_TREND_TARGET_RATIO)
    )
    training_ready = bool(
        scope_summary['training_allowed']
        and lift_summary['training_allowed']
        and bc_summary['selected_bc_readiness_gate']
        and trend_gate
    )
    readiness_loss = (
        (0.0 if scope_summary['training_allowed'] else 1000.0)
        + (0.0 if lift_summary['training_allowed'] else 500.0)
        + (0.0 if bc_summary['selected_bc_readiness_gate'] else 100.0)
        + (0.0 if trend_gate else 50.0)
        + float(bc_summary.get('selected_bc_readiness_loss', 0.0))
        + abs(trend_ratio - 0.15) * 10.0
    )

    readiness_rows.append({
        'candidate': candidate,
        'training_ready': training_ready,
        'readiness_loss': float(readiness_loss),
        'scope_decision': scope_summary['decision'],
        'lift_decision': lift_summary['decision'],
        'bc_decision': bc_summary['decision'],
        'scope_gate': bool(scope_summary['training_allowed']),
        'lift_gate': bool(lift_summary['training_allowed']),
        'bc_readiness_gate': bool(bc_summary['selected_bc_readiness_gate']),
        'trend_gate': trend_gate,
        'trend_target_rows': trend_rows,
        'trend_target_ratio': trend_ratio,
        'wf2020_trend_target_rows': wf2020_trend_rows,
        'trend_fold_counts_json': json.dumps(json_safe(fold_counts), sort_keys=True),
        'selected_unique_actions': int(bc_summary.get('selected_unique_actions', 0)),
        'selected_dominant_action_share': float(bc_summary.get('selected_dominant_action_share', 1.0)),
        'selected_samples': int(bc_summary.get('selected_samples', 0)),
        'current_positive_rebound_lift_rows': int(lift_summary.get('current_positive_rebound_lift_rows', 0)),
        'candidate_release_positive_lift_rows': int(lift_summary.get('candidate_release_positive_lift_rows', 0)),
        'eval_overlap_rows': int(scope_summary.get('eval_overlap_rows', 0)),
    })

    scope_rows.append(scope_audit)
    lift_rows.append(lift_audit)
    label_rows.append(profile_dist)
    action_count_rows.append(action_counts)
    bc_rows.append(bc_scores)
    position_rows.append(position_distribution(targets, candidate=candidate))
    count_rows.append(counts)

feature_overlap_audit = pd.concat(count_rows, ignore_index=True)
target_scope_audit = pd.concat(scope_rows, ignore_index=True)
target_lift_audit = pd.concat(lift_rows, ignore_index=True)
target_lift_summary = pd.DataFrame(lift_summary_rows)
profile_label_comparison = pd.concat(label_rows, ignore_index=True)
action_label_counts = pd.concat(action_count_rows, ignore_index=True)
bc_readiness_scores = pd.concat(bc_rows, ignore_index=True)
target_position_distribution = pd.concat(position_rows, ignore_index=True)
readiness_scores = pd.DataFrame(readiness_rows).sort_values(
    ['training_ready', 'readiness_loss', 'candidate'],
    ascending=[False, True, True],
).reset_index(drop=True)

feature_overlap_audit.to_csv(CARD / 'feature_overlap_audit.csv', index=False)
feature_overlap_audit.to_csv(CARD / 'trend_target_counts.csv', index=False)
target_scope_audit.to_csv(CARD / 'target_scope_audit.csv', index=False)
target_lift_audit.to_csv(CARD / 'target_lift_audit.csv', index=False)
target_lift_summary.to_csv(CARD / 'target_lift_summary.csv', index=False)
profile_label_comparison.to_csv(CARD / 'profile_label_comparison.csv', index=False)
action_label_counts.to_csv(CARD / 'action_label_counts.csv', index=False)
bc_readiness_scores.to_csv(CARD / 'bc_readiness_scores.csv', index=False)
target_position_distribution.to_csv(CARD / 'target_position_distribution.csv', index=False)
readiness_scores.to_csv(CARD / 'readiness_scores.csv', index=False)
readiness_scores

,candidate,training_ready,readiness_loss,scope_decision,lift_decision,bc_decision,scope_gate,lift_gate,bc_readiness_gate,trend_gate,trend_target_rows,trend_target_ratio,wf2020_trend_target_rows,trend_fold_counts_json,selected_unique_actions,selected_dominant_action_share,selected_samples,current_positive_rebound_lift_rows,candidate_release_positive_lift_rows,eval_overlap_rows
0,wf2020_lagged_trend_participation_floor,True,0.191236,state_objective_targets_training_ready,state_objective_target_lift_training_ready,state_objective_bc_ready_for_training,True,True,True,True,631,0.169124,357,"{""wf_2020_train"": 357, ""wf_2022_train"": 136, ""...",5,0.439860,3417,1509,2034,0
1,base_release16_joint_contract,False,51.500000,state_objective_targets_training_ready,state_objective_target_lift_training_ready,state_objective_bc_ready_for_training,True,True,True,False,0,0.000000,0,"{""wf_2020_train"": 0, ""wf_2022_train"": 0, ""wf_2...",5,0.401229,3417,1376,1901,0


In [6]:
selected = readiness_scores.iloc[0].to_dict() if not readiness_scores.empty else {}
selected_candidate = str(selected.get('candidate', ''))
selected_targets = candidate_targets.get(selected_candidate, pd.DataFrame()).copy()
if not selected_targets.empty:
    selected_targets.to_csv(CARD / 'selected_trend_target_contract.csv', index=False)

training_allowed = bool(selected.get('training_ready', False))
if training_allowed:
    decision = 'release16_wf2020_foldlocal_trend_target_ready_for_bc_screen_no_promotion'
    next_mechanism = 'run_release16_wf2020_trend_target_bc_checkpoint_official_three_fold_replay_with_drawdown_drift_stop'
    blocker = 'Target readiness passed, but the recipe was selected from an OOS counterfactual and still requires official three-fold replay before actor retention.'
else:
    decision = 'release16_wf2020_foldlocal_trend_target_blocked_no_training_no_promotion'
    next_mechanism = 'revise_trend_target_recipe_or_find_foldlocal_train_state_before_bc'
    blocker = 'The transferred trend target did not jointly pass fold-local scope, lift, policy label, and wf2020 trend-row sanity gates.'

fold_counts = {}
if selected.get('trend_fold_counts_json'):
    fold_counts = json.loads(str(selected['trend_fold_counts_json']))

selection = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': decision,
    'selected_candidate': selected_candidate,
    'selected_readiness': selected,
    'training_allowed_from_this_card': training_allowed,
    'bc_screen_allowed_next': training_allowed,
    'promotion_allowed': False,
    'actor_retention_allowed': False,
    'target_contract_gate': bool(selected.get('scope_gate', False) and selected.get('lift_gate', False) and selected.get('trend_gate', False)),
    'bc_readiness_gate': bool(selected.get('bc_readiness_gate', False)),
    'target_scope': TARGET_SCOPE,
    'profile': PROFILE_NAME,
    'target_max_dd_pct': TARGET_MAX_DD_PCT,
    'source_gap_card': 'release16_wf2020_trend_participation_gap_v1',
    'source_target_contract': 'release16_joint_drawdown_release_target_contract_v1/joint_target_contract.csv',
    'best_lagged_counterfactual': best_counterfactual,
    'trend_target_fold_counts': fold_counts,
    'counterfactual_oos_config_requires_official_replay': True,
    'counterfactual_equity_guard_transferred_to_target': False,
    'downstream_replay_max_equity_drawdown_guard': best_counterfactual.get('max_equity_drawdown'),
    'primary_blocker': blocker,
    'next_mechanism': next_mechanism,
    'sidecars': SIDE_CARS,
}
summary = {
    **selection,
    'direct_comparison_baseline': 'release16_joint_drawdown_release_target_contract_v1',
    'canonical_baseline': 'soxlbacktest_qlib_walkforward_baseline_v1',
    'role': 'diagnose',
    'changed_layer': ['training_target', 'model_selection', 'bc_handoff'],
    'source_gap_decision': gap_summary.get('decision'),
    'source_gap_primary_blocker': gap_summary.get('primary_blocker'),
    'source_lagged_gate_pass_count': gap_summary.get('lagged_gate_pass_count'),
    'source_same_day_gate_pass_count': gap_summary.get('same_day_gate_pass_count'),
    'conclusion': [
        'The best wf2020 lagged trend participation counterfactual was converted into fold-local target labels, not a runtime overlay.',
        'Labels are generated only inside fold-local train windows and pass the scope audit with zero eval overlap.',
        'BC readiness uses the actual sparse_release_bc_ppo_screen_v1 action mapping instead of a 5 percent bucket proxy.',
        'Promotion and actor retention remain blocked until a downstream official three-fold replay proves the target survives walk-forward and MaxDD <= 20 percent.',
    ],
}

write_json(CARD / 'selection.json', selection)
write_json(CARD / 'summary.json', summary)
conclusion_md = f"""# {EXPERIMENT}

Decision: `{decision}`.

Selected candidate: `{selected_candidate}`.

Target readiness: `{training_allowed}`. Promotion allowed: `False`. Actor retention allowed: `False`.

Key evidence:

- wf2020 trend target rows: `{selected.get('wf2020_trend_target_rows', 0)}`.
- total trend target rows: `{selected.get('trend_target_rows', 0)}`.
- actual policy-label samples: `{selected.get('selected_samples', 0)}`.
- actual unique actions: `{selected.get('selected_unique_actions', 0)}`.
- dominant action share: `{selected.get('selected_dominant_action_share', 1.0)}`.
- eval overlap rows: `{selected.get('eval_overlap_rows', 0)}`.

Risk note: the recipe parameters came from an OOS wf2020 lagged counterfactual. This card only authorizes a downstream BC/checkpoint official replay screen; it does not retain or promote an actor. The counterfactual `max_equity_drawdown={best_counterfactual.get('max_equity_drawdown')}` guard is kept for downstream replay and is not transferred into row labels.

Next action: `{next_mechanism}`.
"""
(CARD / 'conclusion.md').write_text(conclusion_md, encoding='utf-8')
print(json.dumps(selection, indent=2, default=str))

{
  "experiment": "release16_wf2020_foldlocal_trend_target_v1",
  "created_at": "2026-06-15T04:46:40.224509+00:00",
  "decision": "release16_wf2020_foldlocal_trend_target_ready_for_bc_screen_no_promotion",
  "selected_candidate": "wf2020_lagged_trend_participation_floor",
  "selected_readiness": {
    "candidate": "wf2020_lagged_trend_participation_floor",
    "training_ready": true,
    "readiness_loss": 0.19123559367461823,
    "scope_decision": "state_objective_targets_training_ready",
    "lift_decision": "state_objective_target_lift_training_ready",
    "bc_decision": "state_objective_bc_ready_for_training",
    "scope_gate": true,
    "lift_gate": true,
    "bc_readiness_gate": true,
    "trend_gate": true,
    "trend_target_rows": 631,
    "trend_target_ratio": 0.16912355936746182,
    "wf2020_trend_target_rows": 357,
    "trend_fold_counts_json": "{\"wf_2020_train\": 357, \"wf_2022_train\": 136, \"wf_2023_2024_train\": 138}",
    "selected_unique_actions": 5,
    "selected_domi

In [7]:
selection = json.loads((CARD / 'selection.json').read_text(encoding='utf-8'))
readiness_scores = pd.read_csv(CARD / 'readiness_scores.csv')
summary_md = f"""
## Runtime Conclusion

Decision: `{selection['decision']}`. Promotion is blocked.

Selected target candidate: `{selection['selected_candidate']}`. BC/checkpoint screen allowed next: `{selection['bc_screen_allowed_next']}`.

The target labels are fold-local, but the parameter recipe is OOS-informed by `release16_wf2020_trend_participation_gap_v1`; the next step must be official three-fold replay with the `20%` MaxDD guard.

Top readiness row:

```json
{json.dumps(selection['selected_readiness'], indent=2)}
```

Next action: `{selection['next_mechanism']}`.
""".strip()
display(Markdown(summary_md))
readiness_scores

## Runtime Conclusion

Decision: `release16_wf2020_foldlocal_trend_target_ready_for_bc_screen_no_promotion`. Promotion is blocked.

Selected target candidate: `wf2020_lagged_trend_participation_floor`. BC/checkpoint screen allowed next: `True`.

The target labels are fold-local, but the parameter recipe is OOS-informed by `release16_wf2020_trend_participation_gap_v1`; the next step must be official three-fold replay with the `20%` MaxDD guard.

Top readiness row:

```json
{
  "bc_decision": "state_objective_bc_ready_for_training",
  "bc_readiness_gate": true,
  "candidate": "wf2020_lagged_trend_participation_floor",
  "candidate_release_positive_lift_rows": 2034,
  "current_positive_rebound_lift_rows": 1509,
  "eval_overlap_rows": 0,
  "lift_decision": "state_objective_target_lift_training_ready",
  "lift_gate": true,
  "readiness_loss": 0.19123559367461823,
  "scope_decision": "state_objective_targets_training_ready",
  "scope_gate": true,
  "selected_dominant_action_share": 0.43985952589991223,
  "selected_samples": 3417,
  "selected_unique_actions": 5,
  "training_ready": true,
  "trend_fold_counts_json": "{\"wf_2020_train\": 357, \"wf_2022_train\": 136, \"wf_2023_2024_train\": 138}",
  "trend_gate": true,
  "trend_target_ratio": 0.16912355936746182,
  "trend_target_rows": 631,
  "wf2020_trend_target_rows": 357
}
```

Next action: `run_release16_wf2020_trend_target_bc_checkpoint_official_three_fold_replay_with_drawdown_drift_stop`.

,candidate,training_ready,readiness_loss,scope_decision,lift_decision,bc_decision,scope_gate,lift_gate,bc_readiness_gate,trend_gate,trend_target_rows,trend_target_ratio,wf2020_trend_target_rows,trend_fold_counts_json,selected_unique_actions,selected_dominant_action_share,selected_samples,current_positive_rebound_lift_rows,candidate_release_positive_lift_rows,eval_overlap_rows
0,wf2020_lagged_trend_participation_floor,True,0.191236,state_objective_targets_training_ready,state_objective_target_lift_training_ready,state_objective_bc_ready_for_training,True,True,True,True,631,0.169124,357,"{""wf_2020_train"": 357, ""wf_2022_train"": 136, ""...",5,0.439860,3417,1509,2034,0
1,base_release16_joint_contract,False,51.500000,state_objective_targets_training_ready,state_objective_target_lift_training_ready,state_objective_bc_ready_for_training,True,True,True,False,0,0.000000,0,"{""wf_2020_train"": 0, ""wf_2022_train"": 0, ""wf_2...",5,0.401229,3417,1376,1901,0
